In [33]:
import pandas as pd
# Loading the CSV file
df = pd.read_csv(
    "/Users/hd/Desktop/prompt-sensitivity-llms/src/outputs/responses_gemini_20250807_1401.csv"
)

In [34]:
# Step 1: structure sanity check
print(df.shape)
print(df.columns.tolist())

(375, 17)
['timestamp', 'run_id', 'order_idx', 'model', 'model_version', 'domain', 'base_id', 'variant', 'prompt', 'full_prompt', 'response', 'err', 'latency_ms', 'char_len', 'word_len', 'token_count_est', 'response_id']


In [35]:
errors = df["err"].unique().tolist()
print("Unique error messages:", errors)

Unique error messages: [nan, '503 Server Error: Service Unavailable for url: https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent']


In [36]:
df["has_error"] = df["err"].notna()
df["has_error"].value_counts()

has_error
False    311
True      64
Name: count, dtype: int64

In [37]:
# Filter only error rows, show err + response
df[df["has_error"]][["err", "response"]]

,err,response
79,503 Server Error: Service Unavailable for url:...,NaN
90,503 Server Error: Service Unavailable for url:...,NaN
93,503 Server Error: Service Unavailable for url:...,NaN
100,503 Server Error: Service Unavailable for url:...,NaN
111,503 Server Error: Service Unavailable for url:...,NaN
...,...,...
353,503 Server Error: Service Unavailable for url:...,NaN
354,503 Server Error: Service Unavailable for url:...,NaN
356,503 Server Error: Service Unavailable for url:...,NaN
368,503 Server Error: Service Unavailable for url:...,NaN


In [38]:
# Breakdown of errors by domain
error_by_domain = (
    df[df["has_error"]].groupby("domain").size().sort_values(ascending=False)
)

# Breakdown of errors by variant
error_by_variant = (
    df[df["has_error"]].groupby("variant").size().sort_values(ascending=False)
)

# Combined breakdown (domain + variant)
error_combo = (
    df[df["has_error"]]
    .groupby(["domain", "variant"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

error_by_domain, error_by_variant, error_combo

(domain
 Environmental Policy    18
 Political Systems       18
 Historical Events       11
 Public Health           10
 Scientific Consensus     7
 dtype: int64,
 variant
 base                  20
 emotion               16
 paraphrase_neutral    16
 formality              6
 tone                   6
 dtype: int64,
                   domain             variant  count
 7      Political Systems             emotion      8
 1   Environmental Policy             emotion      7
 6      Political Systems                base      7
 10         Public Health  paraphrase_neutral      7
 5      Historical Events           formality      6
 3   Environmental Policy                tone      5
 4      Historical Events                base      5
 0   Environmental Policy                base      3
 2   Environmental Policy  paraphrase_neutral      3
 8      Political Systems  paraphrase_neutral      3
 9          Public Health                base      3
 13  Scientific Consensus  paraphrase_neutral  

In [39]:
# Count errors per run
error_by_run = df[df["has_error"]].groupby("run_id").size()

# Count errors per run + domain
error_by_run_domain = (
    df[df["has_error"]].groupby(["run_id", "domain"]).size().reset_index(name="count")
)

error_by_run, error_by_run_domain.sort_values(
    ["run_id", "count"], ascending=[True, False]
)

(run_id
 2    10
 3    13
 4    21
 5    20
 dtype: int64,
     run_id                domain  count
 0        2  Environmental Policy      3
 2        2     Political Systems      3
 3        2         Public Health      2
 1        2     Historical Events      1
 4        2  Scientific Consensus      1
 7        3     Political Systems      5
 5        3  Environmental Policy      4
 6        3     Historical Events      2
 8        3         Public Health      1
 9        3  Scientific Consensus      1
 10       4  Environmental Policy      5
 11       4     Historical Events      5
 12       4     Political Systems      5
 13       4         Public Health      4
 14       4  Scientific Consensus      2
 15       5  Environmental Policy      6
 17       5     Political Systems      5
 16       5     Historical Events      3
 18       5         Public Health      3
 19       5  Scientific Consensus      3)